<a href="https://colab.research.google.com/github/ornab74/lightcap/blob/main/Grok_Code_Agentized_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# @title Install Dependencies – Fixed & Pinned Versions (no rfc3987 crash)
!pip uninstall -y chromadb  # remove any broken version
!pip install -q --no-deps \
  sentence-transformers \
  httpx \
  tqdm \
  ipywidgets \
  psutil \
  pennylane \
  nest-asyncio \
  "chromadb==0.4.24"  # pinned to stable version before major URI validator changes

# Optional: if you still want the very latest Chroma but safer
# !pip install -q "chromadb>=0.5.0,<0.6.0" --no-deps

print("✅ All packages installed successfully (Chroma pinned to avoid rfc3987 crash)")
print("Next: Run the main GrokForge code cell.")

Found existing installation: chromadb 0.4.24
Uninstalling chromadb-0.4.24:
  Successfully uninstalled chromadb-0.4.24
✅ All packages installed successfully (Chroma pinned to avoid rfc3987 crash)
Next: Run the main GrokForge code cell.


In [ ]:

# =============================================================================
# GROKFORGE ULTIMATE — COLOR-QUBIT + LIGHTNING.GPU (Action-Token Coherence Edition)
# =============================================================================
# ✅ Uses PennyLane lightning.gpu for the Color-Qubit Wheel (8 modes, 20 qubits)
# ✅ QSYNC is driven by the color wheel (qvec) each iteration
# ✅ Grok agent prompting uses [action:tool:NAME]{...}[/action] tool tokens
# ✅ Prints every agent reply + tool execution + loop debug
# ✅ Includes SAFE_MODE, deterministic seeding, GPU sanity checks, micro-benchmarks
# =============================================================================

# ---------------------------------------
# 0) Install + GPU check
# ---------------------------------------
print("Installing packages...")
!pip install --upgrade -q pip setuptools wheel
!pip install -q pennylane "pennylane-lightning[gpu]" custatevec-cu12 psutil tqdm nest_asyncio ipywidgets httpx
# Optional local RAG deps (safe if you want them)
!pip install -q chromadb sentence-transformers || true

import os, json, re, hashlib, time, math, gc, asyncio
from datetime import datetime
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Any, Optional, Tuple

import numpy as np
import psutil
import nest_asyncio
nest_asyncio.apply()

import httpx

import pennylane as qml
from pennylane import numpy as pnp

print("PennyLane:", qml.__version__)
!nvidia-smi --query-gpu=name,memory.total,utilization.gpu --format=csv

# =============================================================================
# 1) CONFIG
# =============================================================================
@dataclass
class GrokForgeConfig:
    # Grok/xAI API
    grok_model: str = "grok-4.1"
    api_key: str = ""
    base_url: str = "https://api.x.ai/v1"
    temperature: float = 0.71
    top_p: float = 0.93
    max_tokens_per_call: int = 8192

    # Color-Qubit wheel
    n_qubits: int = 20
    color_modes: int = 8

    # QLRL memory
    qlrl_shards: int = 12
    qlrl_max_chars: int = 1800
    qlrl_decay: float = 0.89
    max_memory_bank: int = 300

    # Safety/perf
    deterministic: bool = True
    seed: int = 1337
    safe_mode: bool = False             # auto-enabled if GPU memory is low
    safe_mode_mem_gb_threshold: float = 6.0  # if GPU total memory < this, force SAFE_MODE

    # Loop controls
    mode: str = "quantum-max"
    max_iter_quantum_max: int = 10
    max_iter_fast: int = 6

    # Bench gates (tune for your colab GPU)
    forward_time_gate_sec: float = 0.45
    mem_growth_gate_pct: float = 25.0

    colab_mode: bool = True

CFG = GrokForgeConfig()
BASE_LOCAL_PATH = Path("/content/grokforge")
BASE_LOCAL_PATH.mkdir(exist_ok=True, parents=True)

# =============================================================================
# 2) GPU + SAFE MODE detection
# =============================================================================
def _get_gpu_mem_total_gb() -> Optional[float]:
    try:
        import subprocess, shlex
        cmd = "nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits"
        out = subprocess.check_output(shlex.split(cmd)).decode().strip().splitlines()
        if not out:
            return None
        # take first GPU
        mem_mb = float(out[0])
        return mem_mb / 1024.0
    except Exception:
        return None

gpu_total_gb = _get_gpu_mem_total_gb()
if gpu_total_gb is not None:
    print(f"[GPU] total memory ≈ {gpu_total_gb:.2f} GiB")
    if gpu_total_gb < CFG.safe_mode_mem_gb_threshold:
        CFG.safe_mode = True
        print("⚠️ SAFE_MODE auto-enabled due to low GPU memory.")
else:
    print("⚠️ Could not read GPU memory; leaving SAFE_MODE as configured.")

# =============================================================================
# 3) DEVICE (Lightning GPU preferred)
# =============================================================================
def make_quantum_device(n_wires: int):
    try:
        dev = qml.device("lightning.gpu", wires=n_wires, batch_obs=True)
        print(f"✅ Using device: lightning.gpu (wires={n_wires})")
        return dev
    except Exception as e:
        print(f"⚠️ lightning.gpu init failed: {e}")
        dev = qml.device("default.qubit", wires=n_wires)
        print(f"✅ Fallback device: default.qubit (wires={n_wires})")
        return dev

COLOR_DEV = make_quantum_device(CFG.n_qubits)

# =============================================================================
# 4) COLOR-QUBIT WHEEL (8 UNIQUE CIRCUITS)
# =============================================================================
def ring_entangler(n: int):
    for i in range(n - 1):
        qml.CNOT(wires=[i, i + 1])
    qml.CNOT(wires=[n - 1, 0])

def color_block(block_params, mode: int, n: int, safe_mode: bool):
    # block_params shape: (n,)
    for i in range(n):
        theta = block_params[i]
        if mode == 0:      # RED
            qml.RX(theta, wires=i)
        elif mode == 1:    # GREEN
            qml.RY(theta, wires=i)
        elif mode == 2:    # BLUE
            qml.RZ(theta, wires=i)
        elif mode == 3:    # CYAN (RX+RY)
            qml.RX(theta, wires=i); qml.RY(theta, wires=i)
        elif mode == 4:    # MAGENTA (RX+RZ)
            qml.RX(theta, wires=i); qml.RZ(theta, wires=i)
        elif mode == 5:    # YELLOW (RY+RZ)
            qml.RY(theta, wires=i); qml.RZ(theta, wires=i)
        elif mode == 6:    # WHITE (XYZ)
            qml.RX(theta, wires=i); qml.RY(theta, wires=i); qml.RZ(theta, wires=i)
        elif mode == 7:    # BLACK (phase-heavy)
            qml.RZ(theta, wires=i)

    ring_entangler(n)

    # extra phase entanglement only if not safe_mode
    if mode == 7 and (not safe_mode):
        for i in range(0, n, 2):
            qml.ControlledPhaseShift(np.pi / 4, wires=[i, (i + 1) % n])

@qml.qnode(COLOR_DEV, diff_method="adjoint")
def color_wheel_surface(color_params, mix_weights, cross_mix, safe_mode_flag):
    """
    color_params: (modes*n,)
    mix_weights:  (3,)
    cross_mix:    (3,)  -- cross-mode mixer layer (single layer)
    safe_mode_flag: scalar bool-ish (0/1) to keep signature stable for qnode
    returns: expval Z per qubit (n,)
    """
    n = CFG.n_qubits
    modes = CFG.color_modes
    safe_mode = bool(int(safe_mode_flag))

    offset = 0
    for mode in range(modes):
        block = color_params[offset: offset + n]
        color_block(block, mode, n, safe_mode=safe_mode)
        offset += n

    # weighted global mixing layer
    for i in range(n):
        k = (i + 1)
        qml.RX(mix_weights[0] * k, wires=i)
        qml.RY(mix_weights[1] * k, wires=i)
        qml.RZ(mix_weights[2] * k, wires=i)

    ring_entangler(n)

    # single cross-mode mixer (allowed)
    # (kept light in SAFE_MODE)
    for i in range(n):
        k = (i + 1)
        qml.RX(cross_mix[0] * (k if not safe_mode else 0.5 * k), wires=i)
        qml.RY(cross_mix[1] * (k if not safe_mode else 0.5 * k), wires=i)
        qml.RZ(cross_mix[2] * (k if not safe_mode else 0.5 * k), wires=i)

    if not safe_mode:
        ring_entangler(n)

    return [qml.expval(qml.PauliZ(i)) for i in range(n)]

# =============================================================================
# 5) COLOR SYSTEM -> QVEC (QSYNC)
# =============================================================================
class ColorQubitSystem:
    """
    Produces qvec each iteration from the color wheel outputs + features.
    qvec dimension = n_qubits + feature_count (small but rich).
    """
    def __init__(self, n_qubits: int, modes: int, deterministic: bool, seed: int):
        self.n = n_qubits
        self.modes = modes
        self.deterministic = deterministic
        self.seed = seed

        self.rs = np.random.RandomState(seed) if deterministic else np.random.RandomState()
        self.base_color_params = self.rs.randn(self.modes * self.n).astype(np.float64)
        self.base_mix = self.rs.randn(3).astype(np.float64)
        self.base_cross = self.rs.randn(3).astype(np.float64)

    def _iter_nudges(self, entropy: float, cpu: float, ram: float, iteration: int):
        t = float(iteration)
        cpu_n = float(cpu) / 100.0
        ram_n = float(ram) / 100.0

        # small bounded nudges (stable)
        n0 = 0.35 * np.sin(t * 0.11 + entropy * 7.0)
        n1 = 0.20 * np.cos(t * 0.07 + cpu_n * 9.0)
        n2 = 0.15 * np.sin(t * 0.05 + ram_n * 6.0)
        return (n0 + n1 + n2)

    def make_iter_params(self, entropy: float, cpu: float, ram: float, iteration: int, safe_mode: bool):
        nudges = self._iter_nudges(entropy, cpu, ram, iteration)

        # deterministic param update
        color_params = self.base_color_params + nudges
        mix_weights = self.base_mix + np.array([
            0.25 * np.sin(entropy * 8.0 + iteration * 0.03),
            0.20 * np.cos((cpu / 100.0) * 7.0 + iteration * 0.02),
            0.15 * np.sin((ram / 100.0) * 6.0 + iteration * 0.01),
        ], dtype=np.float64)

        cross_mix = self.base_cross + np.array([
            0.12 * np.cos(entropy * 9.0 + iteration * 0.05),
            0.10 * np.sin((cpu / 100.0) * 8.0 + iteration * 0.04),
            0.08 * np.cos((ram / 100.0) * 5.0 + iteration * 0.03),
        ], dtype=np.float64)

        # SAFE_MODE dampening
        if safe_mode:
            color_params = color_params * 0.85
            mix_weights = mix_weights * 0.85
            cross_mix = cross_mix * 0.85

        return (
            pnp.array(color_params, requires_grad=False),
            pnp.array(mix_weights, requires_grad=False),
            pnp.array(cross_mix, requires_grad=False),
        )

    @staticmethod
    def _entropy_proxy(vec: np.ndarray) -> float:
        # p_i = softmax(|vec|)
        x = np.abs(vec).astype(np.float64)
        x = x - np.max(x)
        p = np.exp(x)
        p = p / (np.sum(p) + 1e-12)
        h = -np.sum(p * np.log2(p + 1e-12))
        return float(h)

    @staticmethod
    def _cosine(a: np.ndarray, b: np.ndarray) -> float:
        na = np.linalg.norm(a) + 1e-12
        nb = np.linalg.norm(b) + 1e-12
        return float(np.dot(a, b) / (na * nb))

    def qvec(self, entropy: float, cpu: float, ram: float, iteration: int, safe_mode: bool, prev_qvec: Optional[np.ndarray]):
        cparams, mix, cross = self.make_iter_params(entropy, cpu, ram, iteration, safe_mode)

        z = np.array(color_wheel_surface(cparams, mix, cross, int(safe_mode)), dtype=np.float64)  # (n,)
        # compact but rich features
        feats = np.array([
            float(np.mean(z)),
            float(np.std(z)),
            float(np.mean(np.abs(z))),
            float(np.max(z) - np.min(z)),
            float(self._entropy_proxy(z)),
        ], dtype=np.float64)

        qv = np.concatenate([z, feats], axis=0)

        stability = None
        if prev_qvec is not None and len(prev_qvec) == len(qv):
            stability = self._cosine(qv, prev_qvec)

        return qv, stability

COLOR_SYSTEM = ColorQubitSystem(CFG.n_qubits, CFG.color_modes, CFG.deterministic, CFG.seed)

# =============================================================================
# 6) QUANTUMCORE (QLRL + temp nudge)
# =============================================================================
GLOBAL_MEMORY_BANK: List[str] = []

class QuantumCore:
    def __init__(self):
        self.prev_qvec: Optional[np.ndarray] = None

    def get_entropy_seed(self):
        cpu = psutil.cpu_percent(interval=0.05)
        ram = psutil.virtual_memory().percent
        entropy = ((cpu + ram + os.getpid() + time.time()) % 2000) / 2000
        return float(entropy), float(cpu), float(ram)

    def format_qvec(self, qvec: np.ndarray) -> str:
        head = qvec[: min(24, len(qvec))]
        return "QCOLOR[v2]:" + " | ".join(f"{v:+.3f}" for v in head)

    def qstate_hash_weights(self, qvec: np.ndarray, k: int = 12):
        base = np.tanh(qvec.astype(np.float64))
        bins = np.zeros(k, dtype=np.float64)
        for i, val in enumerate(base):
            bins[i % k] += float(val)
        exps = np.exp(bins - np.max(bins))
        s = float(np.sum(exps)) or 1.0
        return (exps / s).tolist()

    def tri_gram_set(self, s: str):
        s = re.sub(r"\s+", " ", s.strip())
        toks = s.split()
        if len(toks) < 3:
            return set()
        return {" ".join(toks[i:i+3]) for i in range(len(toks) - 2)}

    def qlrl_select_capsules(self, qvec: np.ndarray):
        weights = self.qstate_hash_weights(qvec, k=CFG.qlrl_shards)
        B = len(GLOBAL_MEMORY_BANK)
        if B == 0:
            return []

        candidates = []
        for w in weights:
            stride = max(1, int(4 + 14 * w))
            base = int(B * w) % B
            for j in range(5):
                candidates.append((base + j * stride) % B)
        candidates = list(dict.fromkeys(candidates))

        pivot = GLOBAL_MEMORY_BANK[-1] if GLOBAL_MEMORY_BANK else ""
        pset = self.tri_gram_set(pivot)

        scored = []
        for i in candidates:
            t = GLOBAL_MEMORY_BANK[i]
            if not t:
                continue
            iset = self.tri_gram_set(t)
            jacc = len(pset & iset) / max(1, len(pset | iset))
            scored.append((jacc * weights[i % len(weights)], i))

        scored.sort(reverse=True)
        return [GLOBAL_MEMORY_BANK[i][:CFG.qlrl_max_chars] for (_, i) in scored[:CFG.qlrl_shards]]

    def build_qlrl_capsules(self, qvec: np.ndarray):
        caps = self.qlrl_select_capsules(qvec)
        if not caps:
            return "[no QLRL capsules]"
        scored = []
        for c in caps:
            try:
                idx = len(GLOBAL_MEMORY_BANK) - GLOBAL_MEMORY_BANK[::-1].index(c) - 1
            except ValueError:
                idx = 0
            w = CFG.qlrl_decay ** max(0, len(GLOBAL_MEMORY_BANK) - idx)
            scored.append((w, c))
        scored.sort(reverse=True, key=lambda x: x[0])
        return "\n───\n".join([f"(q-w={w:.4f}) {c}" for w, c in scored])

    def entropic_loop_metrics(self, iteration: int = 0):
        entropy, cpu, ram = self.get_entropy_seed()

        qvec, stability = COLOR_SYSTEM.qvec(
            entropy=entropy, cpu=cpu, ram=ram,
            iteration=iteration,
            safe_mode=CFG.safe_mode,
            prev_qvec=self.prev_qvec
        )
        self.prev_qvec = qvec

        qstr = self.format_qvec(qvec)

        # gain + temp nudge
        entropic_gain = (
            entropy * 0.48
            + (cpu / 100.0) * 0.28
            + (ram / 100.0) * 0.22
            + float(np.var(qvec)) * 0.15
        )
        temp_nudge = -0.07 + entropic_gain * 0.18

        # visualization summary
        top = np.argsort(np.abs(qvec))[-8:]
        viz = "Color-Qvec (top 8 by |value|):\n" + "\n".join(
            f"idx={int(i):03d} val={qvec[i]:+.4f}" for i in top
        )

        return {
            "entropy": entropy, "cpu": cpu, "ram": ram,
            "qvec": qvec, "qstr": qstr, "viz": viz,
            "entropic_gain": float(entropic_gain),
            "temp_nudge": float(temp_nudge),
            "stability": stability,
            "iteration": iteration
        }

QUANTUM = QuantumCore()

# =============================================================================
# 7) TOOL FORGE (for [action:tool:NAME] tokens)
# =============================================================================
class ToolForge:
    def __init__(self):
        self.tools: Dict[str, Dict[str, Any]] = {}

    def register(self, name: str, func, desc: str):
        self.tools[name] = {"func": func, "desc": desc}

    async def execute(self, call: Dict[str, Any]) -> str:
        name = call.get("name")
        args = call.get("args", {}) or {}
        if name not in self.tools:
            return f"[tool_error] Unknown tool: {name}"
        fn = self.tools[name]["func"]
        try:
            if asyncio.iscoroutinefunction(fn):
                return str(await fn(**args))
            else:
                return str(fn(**args))
        except Exception as e:
            return f"[tool_error:{name}] {type(e).__name__}: {e}"

TOOL_FORGE = ToolForge()

# -----------------------
# Tool: summa_context
# -----------------------
async def tool_summa_context(text: str, max_chars: int = 1200):
    prompt = f"Summarize (<= {max_chars} chars) with crisp bullet points:\n\n{text}\n\nSUMMARY:"
    # Use Grok itself but at low temp via grok_generate (defined later)
    # This tool is invoked after grok_generate exists; we patch it in tool registry after definition.
    return "[deferred]"

# -----------------------
# Tool: qmetrics
# -----------------------
def tool_qmetrics():
    qm = QUANTUM.entropic_loop_metrics(iteration=int(time.time()) % 1000)
    return json.dumps({
        "entropy": qm["entropy"],
        "cpu": qm["cpu"],
        "ram": qm["ram"],
        "entropic_gain": qm["entropic_gain"],
        "temp_nudge": qm["temp_nudge"],
        "stability": qm["stability"],
        "qstr_head": qm["qstr"][:120]
    }, indent=2)

# -----------------------
# Tool: micro_benchmark
# -----------------------
def tool_micro_benchmark(reps: int = 2):
    # baseline mem
    p = psutil.Process()
    mem0 = p.memory_info().rss / 1024**2

    # dummy params
    rs = np.random.RandomState(0)
    cparams = pnp.array(rs.randn(CFG.color_modes * CFG.n_qubits), requires_grad=False)
    mix = pnp.array(rs.randn(3), requires_grad=False)
    cross = pnp.array(rs.randn(3), requires_grad=False)

    # warm-up
    _ = color_wheel_surface(cparams, mix, cross, int(CFG.safe_mode))

    # forward timing
    t0 = time.time()
    for _ in range(reps):
        out = color_wheel_surface(cparams, mix, cross, int(CFG.safe_mode))
    fwd = (time.time() - t0) / max(1, reps)

    mem1 = p.memory_info().rss / 1024**2
    return json.dumps({
        "device": str(COLOR_DEV),
        "safe_mode": CFG.safe_mode,
        "forward_avg_sec": fwd,
        "mem_mib_before": mem0,
        "mem_mib_after": mem1,
        "mem_delta_mib": mem1 - mem0,
        "output_sample": [float(x) for x in np.array(out[:6])]
    }, indent=2)

# -----------------------
# Tool: sanitize_action_json
# -----------------------
def tool_sanitize_action_json(text: str):
    # cheap helper if model emits malformed JSON
    # returns best-effort JSON object or raw
    try:
        js = json.loads(text)
        return json.dumps(js, indent=2)
    except Exception:
        # attempt to extract JSON object
        m = re.search(r"\{.*\}", text, re.DOTALL)
        if m:
            try:
                js = json.loads(m.group(0))
                return json.dumps(js, indent=2)
            except Exception:
                pass
    return text

# Register tools (summa_context patched later after grok_generate exists)
TOOL_FORGE.register("qmetrics", tool_qmetrics, "Return current qsync/entropy metrics as JSON")
TOOL_FORGE.register("micro_benchmark", tool_micro_benchmark, "Run lightning.gpu forward micro-benchmark")
TOOL_FORGE.register("sanitize_action_json", tool_sanitize_action_json, "Fix/pretty-print JSON")

# =============================================================================
# 8) GROK GENERATE (parses [action] tokens and executes tools)
# =============================================================================
ACTION_RE = re.compile(r"\[action:(.+?)\](.+?)\[/action\]", re.DOTALL)

async def grok_generate(prompt: str, temperature=None, max_tokens=None, system=None, qsync=None):
    temperature = temperature if temperature is not None else CFG.temperature
    if qsync:
        temperature = max(0.22, min(1.15, temperature + qsync.get("temp_nudge", 0)))

    system = system or (
        "You are Grok-4.1 inside GrokForge. "
        "Use [action:tool:NAME]{JSON}[/action] to call tools. "
        f"QSTATE: {qsync.get('qstr','none') if qsync else 'none'}"
    )

    payload = {
        "model": CFG.grok_model,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user", "content": prompt},
        ],
        "temperature": temperature,
        "top_p": CFG.top_p,
        "max_tokens": max_tokens or CFG.max_tokens_per_call,
    }
    headers = {"Authorization": f"Bearer {CFG.api_key}", "Content-Type": "application/json"}

    async with httpx.AsyncClient(timeout=180) as client:
        resp = await client.post(f"{CFG.base_url}/chat/completions", json=payload, headers=headers)
        resp.raise_for_status()
        content = resp.json()["choices"][0]["message"]["content"]

    print(f"\n[DEBUG] Grok raw reply (head):\n{content[:700]}...\n{'-'*80}\n", flush=True)

    # Execute tools requested via [action:tool:NAME] ... [/action]
    # We run sequentially so the log is coherent.
    tool_calls = ACTION_RE.findall(content)
    if tool_calls:
        print(f"[DEBUG] Detected {len(tool_calls)} [action] blocks. Executing tools...\n", flush=True)

    for (atype, body) in tool_calls:
        atype = atype.strip()
        body = body.strip()

        if atype.startswith("tool:"):
            tname = atype[5:].strip()

            # Tool body should be JSON, but we do best effort
            args = {}
            if body:
                try:
                    args = json.loads(body)
                except Exception:
                    # allow raw string fallback
                    args = {"text": body}

            print(f"[TOOL] Calling {tname} with args keys={list(args.keys())}", flush=True)
            result = await TOOL_FORGE.execute({"name": tname, "args": args})
            print(f"[TOOL] Result head:\n{result[:700]}\n{'-'*60}\n", flush=True)

            # Append tool result so subsequent parsing/agents can see it
            content += f"\n\n[tool_result:{tname}]\n{result}\n[/tool_result:{tname}]"

    # Save to memory bank
    GLOBAL_MEMORY_BANK.append(content[:CFG.qlrl_max_chars])
    while len(GLOBAL_MEMORY_BANK) > CFG.max_memory_bank:
        GLOBAL_MEMORY_BANK.pop(0)

    return content.strip()

# Now patch in summa_context tool (needs grok_generate)
async def tool_summa_context(text: str, max_chars: int = 1200):
    prompt = f"Summarize (<= {max_chars} chars) with crisp bullet points:\n\n{text}\n\nSUMMARY:"
    out = await grok_generate(prompt, temperature=0.25, max_tokens=700, system="You are a summarizer. Output only the summary.", qsync=None)
    return out[:max_chars]

TOOL_FORGE.register("summa_context", tool_summa_context, "Summarize text via Grok at low temperature")

# =============================================================================
# 9) AGENTS (use [action] tokens deliberately)
# =============================================================================
class ForgeAgent:
    def __init__(self, name, role, base_temp=0.67):
        self.name = name
        self.role = role
        self.base_temp = base_temp

    async def think(self, task: str, qsync: Dict[str, Any], draft_excerpt: str):
        prompt = f"""
ROLE: {self.role}
AGENT: {self.name}

COLOR-QUBIT QSYNC:
- QSTR: {qsync.get('qstr','')}
- STABILITY: {qsync.get('stability')}
- VIZ:
{qsync.get('viz','')}

QLRL CAPSULES:
{qsync.get('qlrl','')}

TASK:
{task}

CURRENT DRAFT EXCERPT (may be partial):
{draft_excerpt}

INSTRUCTIONS:
- Use [action:tool:qmetrics]{{}}[/action] if you need live qsync confirmation.
- Use [action:tool:micro_benchmark]{{"reps":2}}[/action] if you propose circuit/depth changes.
- If you need to condense long text: [action:tool:summa_context]{{"text":"...","max_chars":1200}}[/action]
- Output production-ready content. Prefer concrete patches, tests, and measurable validation.

DELIVERABLE FORMAT:
1) Changes
2) Patch or code cells
3) Validation
4) Risks
"""
        temp = self.base_temp + float(qsync.get("temp_nudge", 0.0))
        reply = await grok_generate(prompt, temperature=temp, qsync=qsync)

        print(f"\n=== {self.name.upper()} REPLY ===\n{reply}\n{'='*110}\n", flush=True)
        return reply

RESEARCHER = ForgeAgent("Researcher", "Deep quantum-aware analysis + measurable hypotheses", base_temp=0.62)
CODER = ForgeAgent("Coder", "Production code + performance gates + clean patches", base_temp=0.66)
EDITOR = ForgeAgent("Editor", "Polish + coherence + remove contradictions", base_temp=0.56)
CRITIC = ForgeAgent("Critic", "Adversarial review + edge cases + security", base_temp=0.58)
TEST_ENGINEER = ForgeAgent("TestEngineer", "Comprehensive tests + regressions", base_temp=0.60)
DOCS_ARCHITECT = ForgeAgent("DocsArchitect", "Runbook + tuning + troubleshooting", base_temp=0.54)

# =============================================================================
# 10) STITCHER (merge agent outputs)
# =============================================================================
async def stitch_outputs(agent_texts: List[str], qsync: Dict[str, Any]):
    stitched_prompt = f"""
You are StitchWeaveAgent. Merge the following agent outputs into ONE cohesive deliverable.

Rules:
- Preserve code blocks exactly.
- Remove duplicate sections.
- Ensure final includes: (A) spec, (B) code, (C) tests, (D) docs, (E) validation plan.
- If any agent used tool results, incorporate them.

AGENT OUTPUTS:
{("\n\n" + "-"*80 + "\n\n").join(agent_texts)}
"""
    return await grok_generate(stitched_prompt, temperature=0.46, qsync=qsync)

# =============================================================================
# 11) FORGE LOOP (full debug logging)
# =============================================================================
class ForgeLoop:
    async def run(self, task: str):
        print(f"🌌 Starting ForgeLoop ({CFG.mode}) — Color-Qubit / lightning.gpu preferred")
        draft = ""
        iteration = 0
        max_iter = CFG.max_iter_quantum_max if CFG.mode == "quantum-max" else CFG.max_iter_fast

        # Baseline benchmark for mem gate
        print("📏 Running baseline micro-benchmark...")
        base_bench = tool_micro_benchmark(reps=2)
        print("[BASELINE BENCH]\n", base_bench, "\n")
        try:
            base_js = json.loads(base_bench)
            base_mem = float(base_js["mem_mib_after"])
        except Exception:
            base_mem = None

        while iteration < max_iter:
            iteration += 1
            print(f"\n=== ITERATION {iteration} START ===\n", flush=True)

            qm = QUANTUM.entropic_loop_metrics(iteration * 11)
            qsync = {
                "qstr": qm["qstr"],
                "temp_nudge": qm["temp_nudge"],
                "qlrl": QUANTUM.build_qlrl_capsules(qm["qvec"]),
                "viz": qm["viz"],
                "stability": qm["stability"],
            }

            print(f"Quantum Metrics: Entropy={qm['entropy']:.4f} | Gain={qm['entropic_gain']:.3f} | TempNudge={qm['temp_nudge']:.3f} | Stability={qm['stability']}")
            print(qsync["viz"][:800] + "\n", flush=True)

            agents = [RESEARCHER, CODER, EDITOR, CRITIC, TEST_ENGINEER, DOCS_ARCHITECT]
            outputs = []
            for ag in agents:
                reply = await ag.think(task=task, qsync=qsync, draft_excerpt=draft[-8000:])
                outputs.append(reply)
                print(f"[DEBUG] {ag.name} contributed {len(reply)} chars\n", flush=True)

            combined = await stitch_outputs(outputs, qsync)
            draft = combined
            print(f"[DEBUG] Draft length now: {len(draft)} chars\n", flush=True)

            # Hard gates (performance + memory growth)
            bench = tool_micro_benchmark(reps=2)
            print("[ITER BENCH]\n", bench, "\n")

            try:
                js = json.loads(bench)
                fwd = float(js["forward_avg_sec"])
                mem_after = float(js["mem_mib_after"])
            except Exception:
                fwd, mem_after = None, None

            if fwd is not None and fwd > CFG.forward_time_gate_sec:
                print(f"⚠️ Gate: forward_avg_sec {fwd:.3f} > {CFG.forward_time_gate_sec:.3f}. Enabling SAFE_MODE.")
                CFG.safe_mode = True

            if base_mem is not None and mem_after is not None:
                growth = 100.0 * (mem_after - base_mem) / max(1e-9, base_mem)
                if growth > CFG.mem_growth_gate_pct:
                    print(f"⚠️ Gate: memory growth {growth:.1f}% > {CFG.mem_growth_gate_pct:.1f}%. Enabling SAFE_MODE + trimming memory bank.")
                    CFG.safe_mode = True
                    # trim memory bank aggressively
                    while len(GLOBAL_MEMORY_BANK) > 150:
                        GLOBAL_MEMORY_BANK.pop(0)

            # Heuristic completion
            if len(draft) > 12000 and iteration >= 2:
                print("🎉 Heuristic COMPLETE: draft is substantial.")
                break

        # Final polish pass
        final = await grok_generate(
            f"""
FinalPolishAgent: Produce the FINAL answer.
Must include:
- A clear spec of the Color-Qubit qvec system
- Full Colab code (cells)
- Tests (pytest + quick pass/fail cell)
- README runbook and troubleshooting
- Validation commands and expected outputs

DRAFT:
{draft}
""",
            temperature=0.26,
            qsync=qsync
        )

        print("\n=== FINAL OUTPUT GENERATED ===\n", flush=True)
        return final

# =============================================================================
# 12) Colab UI / Entry
# =============================================================================
def run_colab_ui():
    print("🎉 GROKFORGE — Action Tokens + Color-Qubit + Lightning GPU (Colab)")
    try:
        from google.colab import userdata, files
        CFG.api_key = userdata.get("GROK_API_KEY")
        if not CFG.api_key:
            raise RuntimeError("Missing GROK_API_KEY")
        print("✅ GROK_API_KEY loaded from Colab Secrets.")
    except Exception:
        print("❌ Add GROK_API_KEY in Colab Secrets (left sidebar → Secrets).")
        return

    print(f"Config: n_qubits={CFG.n_qubits}, safe_mode={CFG.safe_mode}, deterministic={CFG.deterministic}, seed={CFG.seed}")
    task = input("Enter your coding task (press Enter for demo):\n").strip()
    if not task:
        task = "Refactor this repository into a clean Python package with tests, docs, and a Dockerfile."

    project = input("Project name (for output file):\n").strip() or "default"

    async def _go():
        loop = ForgeLoop()
        result = await loop.run(task)
        out_path = BASE_LOCAL_PATH / f"{project}_output_{datetime.utcnow().strftime('%Y%m%d_%H%M')}.md"
        out_path.write_text(result, encoding="utf-8")
        print(f"💾 Saved locally: {out_path}")
        try:
            files.download(str(out_path))
        except Exception:
            pass
        return result

    asyncio.run(_go())

# Auto-run in Colab
if __name__ == "__main__" or "google.colab" in str(__import__("sys").modules):
    if CFG.colab_mode:
        run_colab_ui()
    else:
        print("Run in Colab for best experience.")

Installing packages...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-exporter-otlp-proto-common==1.38.0, but you have opentelemetry-exporter-otlp-proto-common 1.39.1 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-proto==1.38.0, but you have opentelemetry-proto 1.39.1 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-sdk~=1.38.0, but you have opentelemetry-sdk 1.39.1 which is incompatible.
opentelemetry-exporter-gcp-logging 1.11.0a0 requires opentelemetry-sdk<1.39.0,>=1.35.0, but you have opentelemetry-sdk 1.39.1 which is incompatible.
PennyLane: 0.44.0
name, memory.total [MiB], utilization.gpu [%]
Tesla T4, 15360 MiB, 0 %
[GPU] total memory ≈ 15.00 GiB
✅ Using device: lightning.gpu (wires=20)
🎉 GROKFORGE —

HTTPStatusError: Client error '400 Bad Request' for url 'https://api.x.ai/v1/chat/completions'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/400

In [ ]:

# Upgrade base packages first
!pip install --upgrade pip setuptools wheel

# Install PennyLane + Lightning-GPU
!pip install pennylane pennylane-lightning[gpu]

# Install cuQuantum dependencies (NVIDIA provides these)
!pip install custatevec-cu12  # for CUDA 12.x (most common in 2026)
# or custatevec-cu11 for CUDA 11.x

In [ ]:

# =============================================================================
# PENNYLANE GPU TEST SCRIPT — 2026 Colab Edition
# =============================================================================
# Tests lightning.gpu vs lightning.qubit
# Measures time for 16-qubit GHZ-like circuit
# =============================================================================

print("Step 1: Installing / upgrading packages...")
!pip install --upgrade -q pip setuptools wheel
!pip install -q pennylane pennylane-lightning[gpu] custatevec-cu12

print("\nStep 2: Importing and checking versions...")
import pennylane as qml
import time
import numpy as np

print("PennyLane version:", qml.__version__)

# List available device plugins (correct way)
print("\nAvailable device plugins:")
print(list(qml.devices.keys()) if hasattr(qml.devices, 'keys') else qml.devices)

# GPU Info
print("\nGPU hardware:")
!nvidia-smi --query-gpu=name,memory.total --format=csv

# =============================================================================
# GPU DEVICE TEST
# =============================================================================
print("\nStep 3: Trying lightning.gpu (GPU)...")
try:
    dev_gpu = qml.device("lightning.gpu", wires=16)
    print("SUCCESS: lightning.gpu device created!")
    print("Device:", dev_gpu)
    print("Short name:", getattr(dev_gpu, 'short_name', 'N/A'))
except Exception as e:
    print("FAILED to create lightning.gpu:", str(e))
    dev_gpu = None

# =============================================================================
# CPU FAST FALLBACK
# =============================================================================
print("\nStep 4: lightning.qubit (fast CPU fallback)...")
dev_cpu = qml.device("lightning.qubit", wires=16)
print("lightning.qubit created:", dev_cpu)

# =============================================================================
# TEST CIRCUIT — GHZ state on 16 qubits
# =============================================================================
def run_circuit(dev):
    @qml.qnode(dev)
    def circuit():
        for i in range(16):
            qml.Hadamard(wires=i)
            if i < 15:
                qml.CNOT(wires=[i, i+1])
        return qml.state()

    start = time.time()
    state = circuit()
    elapsed = time.time() - start

    print(f"\nCircuit executed on {dev.short_name if hasattr(dev, 'short_name') else dev.name}")
    print(f"Time taken: {elapsed:.4f} seconds")
    print("State length:", len(state))
    print("Sample probs (first 8):", np.abs(state[:8])**2)
    print("Max prob:", np.max(np.abs(state)**2))
    return elapsed

# Run both if GPU available
if dev_gpu:
    print("\n=== GPU RUN ===")
    time_gpu = run_circuit(dev_gpu)

print("\n=== CPU FAST RUN ===")
time_cpu = run_circuit(dev_cpu)

if dev_gpu:
    print(f"\nSpeedup (GPU vs CPU): {time_cpu / time_gpu:.1f}x faster on GPU")
else:
    print("\nGPU not available — only CPU timing shown")

print("\nDone! If lightning.gpu worked → you can use it in GrokForge.")
print("If not → stick with lightning.qubit (still much faster than default.qubit)")

Step 1: Installing / upgrading packages...

Step 2: Importing and checking versions...
PennyLane version: 0.44.0

Available device plugins:
<module 'pennylane.devices' from '/usr/local/lib/python3.12/dist-packages/pennylane/devices/__init__.py'>

GPU hardware:
name, memory.total [MiB]
Tesla T4, 15360 MiB

Step 3: Trying lightning.gpu (GPU)...
SUCCESS: lightning.gpu device created!
Device: <lightning.gpu device (wires=16) at 0x7add4c30b7d0>
Short name: N/A

Step 4: lightning.qubit (fast CPU fallback)...
lightning.qubit created: <lightning.qubit device (wires=16) at 0x7add1c9edf70>

=== GPU RUN ===

Circuit executed on lightning.gpu
Time taken: 0.3439 seconds
State length: 65536
Sample probs (first 8): [1.52587891e-05 1.52587891e-05 1.52587891e-05 1.52587891e-05
 1.52587891e-05 1.52587891e-05 1.52587891e-05 1.52587891e-05]
Max prob: 1.5258789062499963e-05

=== CPU FAST RUN ===

Circuit executed on lightning.qubit
Time taken: 0.0047 seconds
State length: 65536
Sample probs (first 8): [1.5

In [ ]:

# =============================================================================
# PENNYLANE GPU ADVANCED TEST — 16 QUBITS (GPU Stress + Debug)
# =============================================================================
# Tests lightning.gpu with deep circuit, QFT, measurements, gradients
# Should run in <1–3 seconds on Colab T4 GPU
# =============================================================================

print("Installing / upgrading packages...")
!pip install --upgrade -q pip setuptools wheel
!pip install -q pennylane pennylane-lightning[gpu] custatevec-cu12

import pennylane as qml
import time
import numpy as np
import psutil

print("PennyLane version:", qml.__version__)
print("GPU Info:")
!nvidia-smi --query-gpu=name,memory.total,utilization.gpu --format=csv

# =============================================================================
# GPU DEVICE (with batching for speed)
# =============================================================================
try:
    dev = qml.device("lightning.gpu", wires=16, batch_obs=True)
    print("\nSUCCESS: lightning.gpu (GPU) with batch_obs=True")
    print("Device:", dev)
except Exception as e:
    print("\nGPU FAILED:", str(e))
    dev = qml.device("lightning.qubit", wires=16)
    print("Fallback to lightning.qubit (fast CPU)")

# =============================================================================
# ADVANCED CIRCUIT — Variational + Full QFT + Measurements
# =============================================================================
@qml.qnode(dev, diff_method="adjoint")  # adjoint = fast GPU gradients
def advanced_gpu_circuit(params):
    # Variational ansatz (hardware-efficient, 3 layers)
    for layer in range(3):
        for i in range(16):
            qml.RX(params[layer*16 + i], wires=i)
            qml.RZ(params[layer*16 + 16 + i], wires=i)
        for i in range(15):
            qml.CNOT(wires=[i, i+1])
        qml.CNOT(wires=[15, 0])  # ring topology

    # Full QFT on all 16 qubits (very entangling, GPU-heavy)
    for i in range(16):
        for j in range(i+1, 16):
            qml.ControlledPhaseShift(2*np.pi / (2**(j-i)), wires=[i, j])
        qml.Hadamard(wires=i)

    # Measurements (expectation + probs)
    return [
        qml.expval(qml.PauliZ(i) @ qml.PauliZ((i+1)%16)) for i in range(16)
    ], qml.probs(wires=range(8))  # probs on first 8 qubits

# =============================================================================
# RUN & TIME EVERYTHING
# =============================================================================
params = np.random.randn(3*32)  # 3 layers × (RX + RZ per qubit)

print("\n=== Starting GPU circuit benchmark ===")

start_total = time.time()

# Warm-up run
print("Warm-up run...")
_ = advanced_gpu_circuit(params)

# Real timed run
print("Timed run...")
start = time.time()
expvals, probs = advanced_gpu_circuit(params)
end = time.time()

total_time = time.time() - start_total

print(f"\nCircuit execution time: {end - start:.4f} seconds")
print(f"Total time (incl warm-up): {total_time:.4f} seconds")
print(f"Memory usage (approx): {psutil.Process().memory_info().rss / 1024**2:.1f} MiB")

# Results summary
print("\nExpectation values (Z_i Z_{i+1}):")
print(np.round(expvals, 4))

print("\nProbabilities (first 8 qubits) sample:")
print(np.round(probs[:8], 6))

# Gradient test (GPU stress)
print("\nComputing gradient (adjoint diff)...")
start_grad = time.time()
grad = qml.grad(advanced_gpu_circuit)(params)
end_grad = time.time()
print(f"Gradient computation time: {end_grad - start_grad:.4f} seconds")
print("Gradient sample (first 8):", np.round(grad[:8], 4))

print("\n=== Test Complete ===")
print("If times are <1–3s and no errors → GPU is fully working!")
print("Use 'lightning.gpu' in your GrokForge script for 16–24 qubits.")

Installing / upgrading packages...
PennyLane version: 0.44.0
GPU Info:
name, memory.total [MiB], utilization.gpu [%]
Tesla T4, 15360 MiB, 0 %

SUCCESS: lightning.gpu (GPU) with batch_obs=True
Device: <lightning.gpu device (wires=16) at 0x7add1cefc2f0>

=== Starting GPU circuit benchmark ===
Warm-up run...


QuantumFunctionError: Device <lightning.gpu device (wires=16) at 0x7add1cefc2f0> does not support adjoint with requested circuit.

In [ ]:

# =============================================================================
# PENNYLANE GPU ADVANCED TEST — 16 QUBITS (Adjoint Compatible)
# =============================================================================

print("Installing / upgrading packages...")
!pip install --upgrade -q pip setuptools wheel
!pip install -q pennylane pennylane-lightning[gpu] custatevec-cu12

import pennylane as qml
import time
import numpy as np
import psutil

print("PennyLane version:", qml.__version__)
print("GPU Info:")
!nvidia-smi --query-gpu=name,memory.total,utilization.gpu --format=csv

# =============================================================================
# GPU DEVICE
# =============================================================================
try:
    dev = qml.device("lightning.gpu", wires=16, batch_obs=True)
    print("\nSUCCESS: lightning.gpu (GPU)")
    print("Device:", dev)
except Exception as e:
    print("\nGPU FAILED:", str(e))
    dev = qml.device("lightning.qubit", wires=16)
    print("Fallback to lightning.qubit (CPU)")

# =============================================================================
# ADVANCED CIRCUIT — Variational + Full QFT
# (Adjoint-compatible: expectation values only)
# =============================================================================
@qml.qnode(dev, diff_method="adjoint")
def advanced_gpu_circuit(params):

    # 3-layer hardware-efficient ansatz
    for layer in range(3):
        for i in range(16):
            qml.RX(params[layer*32 + i], wires=i)
            qml.RZ(params[layer*32 + 16 + i], wires=i)

        # Ring entanglement
        for i in range(15):
            qml.CNOT(wires=[i, i+1])
        qml.CNOT(wires=[15, 0])

    # Full QFT (heavy entanglement)
    for i in range(16):
        for j in range(i+1, 16):
            qml.ControlledPhaseShift(2*np.pi / (2**(j-i)), wires=[i, j])
        qml.Hadamard(wires=i)

    # Adjoint supports expectation values only
    return [
        qml.expval(qml.PauliZ(i) @ qml.PauliZ((i+1)%16))
        for i in range(16)
    ]

# =============================================================================
# RUN BENCHMARK
# =============================================================================
params = np.random.randn(3 * 32)

print("\n=== Starting GPU circuit benchmark ===")
start_total = time.time()

# Warm-up
print("Warm-up run...")
_ = advanced_gpu_circuit(params)

# Timed forward pass
print("Timed forward pass...")
start = time.time()
expvals = advanced_gpu_circuit(params)
end = time.time()

forward_time = end - start
total_time = time.time() - start_total

print(f"\nForward pass time: {forward_time:.4f} seconds")
print(f"Total time (incl warm-up): {total_time:.4f} seconds")
print(f"Memory usage: {psutil.Process().memory_info().rss / 1024**2:.1f} MiB")

print("\nExpectation values (Z_i Z_{i+1}):")
print(np.round(expvals, 4))

# =============================================================================
# GRADIENT BENCHMARK
# =============================================================================
print("\nComputing gradient (adjoint)...")
start_grad = time.time()
grad = qml.grad(advanced_gpu_circuit)(params)
end_grad = time.time()

print(f"Gradient computation time: {end_grad - start_grad:.4f} seconds")
print("Gradient sample (first 8 params):")
print(np.round(grad[:8], 4))

print("\n=== Test Complete ===")
print("If forward <1s and gradient <2s → GPU is working correctly!")

Installing / upgrading packages...
PennyLane version: 0.44.0
GPU Info:
name, memory.total [MiB], utilization.gpu [%]
Tesla T4, 15360 MiB, 0 %

SUCCESS: lightning.gpu (GPU)
Device: <lightning.gpu device (wires=16) at 0x7add4c30b650>

=== Starting GPU circuit benchmark ===
Warm-up run...
Timed forward pass...

Forward pass time: 0.0222 seconds
Total time (incl warm-up): 0.0560 seconds
Memory usage: 635.5 MiB

Expectation values (Z_i Z_{i+1}):
[-0.0037  0.0464 -0.0652 -0.0612  0.0128  0.1822  0.0258 -0.1171 -0.0562
 -0.0459  0.0586 -0.0546 -0.0039 -0.0142 -0.0062  0.0106]

Computing gradient (adjoint)...
Gradient computation time: 0.0222 seconds
Gradient sample (first 8 params):
[]

=== Test Complete ===
If forward <1s and gradient <2s → GPU is working correctly!


/usr/local/lib/python3.12/dist-packages/pennylane/_grad/grad.py:337: UserWarning: Attempted to differentiate a function with no trainable parameters. If this is unintended, please add trainable parameters via the 'requires_grad' attribute or 'argnums' keyword.
  warnings.warn(


In [ ]:

# =============================================================================
# PENNYLANE GPU STRESS TEST — 20 QUBITS (Adjoint Compatible)
# =============================================================================

print("Installing / upgrading packages...")
!pip install --upgrade -q pip setuptools wheel
!pip install -q pennylane pennylane-lightning[gpu] custatevec-cu12

import pennylane as qml
import time
import numpy as np
import psutil

print("PennyLane version:", qml.__version__)
print("GPU Info:")
!nvidia-smi --query-gpu=name,memory.total,utilization.gpu --format=csv

# =============================================================================
# DEVICE (20 QUBITS)
# =============================================================================
try:
    dev = qml.device("lightning.gpu", wires=20, batch_obs=True)
    print("\nSUCCESS: lightning.gpu (GPU)")
    print("Device:", dev)
except Exception as e:
    print("\nGPU FAILED:", str(e))
    dev = qml.device("lightning.qubit", wires=20)
    print("Fallback to lightning.qubit (CPU)")

# =============================================================================
# ADVANCED 20-QUBIT CIRCUIT
#  - 3-layer hardware efficient ansatz
#  - Ring entanglement
#  - Heavy controlled phase mesh (QFT-style)
#  - Expectation values only (adjoint compatible)
# =============================================================================
@qml.qnode(dev, diff_method="adjoint")
def gpu_20q_circuit(params):

    # 3 layers
    for layer in range(3):
        for i in range(20):
            qml.RX(params[layer*40 + i], wires=i)
            qml.RZ(params[layer*40 + 20 + i], wires=i)

        # Ring entanglement
        for i in range(19):
            qml.CNOT(wires=[i, i+1])
        qml.CNOT(wires=[19, 0])

    # QFT-style heavy entanglement (truncated for stability)
    for i in range(20):
        for j in range(i+1, min(i+6, 20)):  # limit span for stability
            qml.ControlledPhaseShift(
                2*np.pi / (2**(j-i)), wires=[i, j]
            )
        qml.Hadamard(wires=i)

    # Expectation values (adjoint supported)
    return [
        qml.expval(qml.PauliZ(i) @ qml.PauliZ((i+1)%20))
        for i in range(20)
    ]

# =============================================================================
# RUN BENCHMARK
# =============================================================================
params = np.random.randn(3 * 40)

print("\n=== Starting 20-Qubit GPU Benchmark ===")
start_total = time.time()

# Warm-up
print("Warm-up run...")
_ = gpu_20q_circuit(params)

# Timed forward
print("Timed forward pass...")
start = time.time()
expvals = gpu_20q_circuit(params)
end = time.time()

forward_time = end - start
total_time = time.time() - start_total

print(f"\nForward pass time: {forward_time:.4f} seconds")
print(f"Total time (incl warm-up): {total_time:.4f} seconds")
print(f"Memory usage: {psutil.Process().memory_info().rss / 1024**2:.1f} MiB")

print("\nExpectation values sample:")
print(np.round(expvals[:8], 4))

# =============================================================================
# GRADIENT BENCHMARK
# =============================================================================
print("\nComputing gradient (adjoint)...")
start_grad = time.time()
grad = qml.grad(gpu_20q_circuit)(params)
end_grad = time.time()

print(f"Gradient computation time: {end_grad - start_grad:.4f} seconds")
print("Gradient sample (first 8 params):")
print(np.round(grad[:8], 4))

print("\n=== 20-Qubit Test Complete ===")
print("If forward <2–4s and gradient <4–8s → GPU is fully healthy.")

Installing / upgrading packages...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
ERROR: Operation cancelled by user
^C


ModuleNotFoundError: No module named 'pennylane'

In [ ]:

# =============================================================================
# 24-QUBIT LLM-RAG SIMTHEORY CIRCUIT (GPU + ADJOINT)
# =============================================================================

print("Installing packages...")
!pip install --upgrade -q pip setuptools wheel
!pip install -q pennylane pennylane-lightning[gpu] custatevec-cu12

import pennylane as qml
import numpy as np
import time
import psutil

print("PennyLane version:", qml.__version__)
!nvidia-smi --query-gpu=name,memory.total,utilization.gpu --format=csv

# =============================================================================
# DEVICE
# =============================================================================
dev = qml.device("lightning.gpu", wires=24, batch_obs=True)

# =============================================================================
# SIMTHEORY RAG CIRCUIT
# =============================================================================
@qml.qnode(dev, diff_method="adjoint")
def rag_llm_sim_circuit(params):

    # ---------------------------------------------------------
    # BLOCK 1 — TOKEN EMBEDDING (Query + Context Encoding)
    # ---------------------------------------------------------
    for i in range(24):
        qml.RY(params[i], wires=i)
        qml.RZ(params[24 + i], wires=i)

    # ---------------------------------------------------------
    # BLOCK 2 — SELF ATTENTION (Within Query + Context)
    # ---------------------------------------------------------
    # Query self-mixing (0–7)
    for i in range(0, 8):
        qml.CNOT(wires=[i, (i+1)%8])

    # Context self-mixing (8–15)
    for i in range(8, 16):
        qml.CNOT(wires=[i, 8 + (i+1-8)%8])

    # ---------------------------------------------------------
    # BLOCK 3 — CROSS ATTENTION (Query ↔ Context)
    # ---------------------------------------------------------
    for i in range(8):
        qml.CNOT(wires=[i, i+8])
        qml.ControlledPhaseShift(np.pi/4, wires=[i+8, i])

    # ---------------------------------------------------------
    # BLOCK 4 — LATENT FUSION LAYER (16–23)
    # ---------------------------------------------------------
    for i in range(8):
        qml.CNOT(wires=[i, 16+i])
        qml.CNOT(wires=[8+i, 16+i])

    # Dense entanglement inside latent block
    for i in range(16, 24):
        for j in range(i+1, 24):
            if j - i <= 3:  # local dense mixing
                qml.ControlledPhaseShift(np.pi/8, wires=[i, j])
        qml.Hadamard(wires=i)

    # ---------------------------------------------------------
    # BLOCK 5 — DEEP MIXING STACK (Transformer Depth)
    # ---------------------------------------------------------
    depth = 2
    offset = 48
    for d in range(depth):
        for i in range(24):
            qml.RX(params[offset + d*24 + i], wires=i)

        # ring entanglement
        for i in range(23):
            qml.CNOT(wires=[i, i+1])
        qml.CNOT(wires=[23, 0])

    # ---------------------------------------------------------
    # OUTPUT HEAD (Expectation values only)
    # ---------------------------------------------------------
    return [
        qml.expval(qml.PauliZ(i) @ qml.PauliZ((i+1)%24))
        for i in range(24)
    ]

# =============================================================================
# PARAM INITIALIZATION
# =============================================================================
# 24 RY + 24 RZ + 2*24 RX layers
param_count = 24 + 24 + 48
params = np.random.randn(param_count)

print("\n=== 24-Qubit LLM-RAG Simulation Benchmark ===")

# Warm-up
print("Warm-up run...")
_ = rag_llm_sim_circuit(params)

# Forward timing
start = time.time()
expvals = rag_llm_sim_circuit(params)
forward_time = time.time() - start

print(f"\nForward pass time: {forward_time:.4f} sec")
print("Expval sample:", np.round(expvals[:6], 4))

# Gradient timing
print("\nComputing gradient...")
start = time.time()
grad = qml.grad(rag_llm_sim_circuit)(params)
grad_time = time.time() - start

print(f"Gradient time: {grad_time:.4f} sec")
print("Grad sample:", np.round(grad[:6], 4))

print(f"\nMemory usage: {psutil.Process().memory_info().rss/1024**2:.1f} MiB")
print("=== TEST COMPLETE ===")

Installing packages...
PennyLane version: 0.44.0
name, memory.total [MiB], utilization.gpu [%]
Tesla T4, 15360 MiB, 0 %

=== 24-Qubit LLM-RAG Simulation Benchmark ===
Warm-up run...

Forward pass time: 0.4419 sec
Expval sample: [ 0.0001  0.002  -0.      0.0036 -0.0001 -0.0202]

Computing gradient...


/usr/local/lib/python3.12/dist-packages/pennylane/_grad/grad.py:337: UserWarning: Attempted to differentiate a function with no trainable parameters. If this is unintended, please add trainable parameters via the 'requires_grad' attribute or 'argnums' keyword.
  warnings.warn(


Gradient time: 0.4401 sec
Grad sample: []

Memory usage: 623.8 MiB
=== TEST COMPLETE ===


In [ ]:

# =============================================================================
# 24-QUBIT ADVANCED MULTI-HEAD RAG SIMULATION
# =============================================================================

import pennylane as qml
import numpy as np
import time
import psutil

dev = qml.device("lightning.gpu", wires=24, batch_obs=True)

# -----------------------------------------------------------------------------
# Circuit
# -----------------------------------------------------------------------------
@qml.qnode(dev, diff_method="adjoint")
def advanced_rag_attention(params):

    head_size = 6   # 4 heads × 6 qubits = 24
    depth = 4
    idx = 0

    # -------------------------------------------------------------------------
    # INPUT EMBEDDING
    # -------------------------------------------------------------------------
    for i in range(24):
        qml.RY(params[idx], wires=i); idx += 1
        qml.RZ(params[idx], wires=i); idx += 1

    # -------------------------------------------------------------------------
    # MULTI-HEAD ATTENTION BLOCK
    # -------------------------------------------------------------------------
    for h in range(4):
        start = h * head_size
        end = start + head_size

        # self mixing within head
        for i in range(start, end-1):
            qml.CNOT(wires=[i, i+1])

        # trainable attention weight (phase routing)
        for i in range(start, end):
            for j in range(i+1, end):
                qml.ControlledPhaseShift(params[idx], wires=[i, j])
                idx += 1

    # -------------------------------------------------------------------------
    # CROSS-HEAD ROUTING (global attention)
    # -------------------------------------------------------------------------
    for i in range(12):
        qml.CNOT(wires=[i, i+12])

    # -------------------------------------------------------------------------
    # TRANSFORMER DEPTH STACK
    # -------------------------------------------------------------------------
    for d in range(depth):
        for i in range(24):
            qml.RX(params[idx], wires=i); idx += 1

        for i in range(23):
            qml.CNOT(wires=[i, i+1])
        qml.CNOT(wires=[23, 0])

    # -------------------------------------------------------------------------
    # OUTPUT HEAD
    # -------------------------------------------------------------------------
    return [qml.expval(qml.PauliZ(i)) for i in range(24)]


# -----------------------------------------------------------------------------
# Parameter Count Calculation
# -----------------------------------------------------------------------------
embedding_params = 24 * 2
phase_params = 4 * (6*5//2)     # per head full pair mesh
transformer_params = 4 * 24
total_params = embedding_params + phase_params + transformer_params

print("Total trainable parameters:", total_params)

# IMPORTANT: make trainable
params = qml.numpy.random.randn(total_params, requires_grad=True)

print("\n=== ADVANCED 24-QUBIT ATTENTION BENCHMARK ===")

# Warm-up
_ = advanced_rag_attention(params)

# Forward
start = time.time()
out = advanced_rag_attention(params)
forward_time = time.time() - start

print("Forward time:", round(forward_time, 4), "sec")

# Gradient
start = time.time()
grad = qml.grad(advanced_rag_attention)(params)
grad_time = time.time() - start

print("Gradient time:", round(grad_time, 4), "sec")
print("Gradient sample:", np.round(grad[:6], 4))

print("Memory usage:",
      round(psutil.Process().memory_info().rss/1024**2, 1), "MiB")

Total trainable parameters: 204

=== ADVANCED 24-QUBIT ATTENTION BENCHMARK ===


In [ ]:

# =============================================================================
# 24-QUBIT ADVANCED MULTI-HEAD RAG SIMULATION (GPU + ADJOINT)
# =============================================================================

print("Installing / upgrading packages...")
!pip install --upgrade -q pip setuptools wheel
!pip install -q pennylane pennylane-lightning[gpu] custatevec-cu12

import pennylane as qml
import numpy as np
import time
import psutil

print("PennyLane version:", qml.__version__)
print("GPU Info:")
!nvidia-smi --query-gpu=name,memory.total,utilization.gpu --format=csv

# =============================================================================
# DEVICE SETUP
# =============================================================================
dev = qml.device("lightning.gpu", wires=24, batch_obs=True)
print("\nUsing device:", dev)

# =============================================================================
# ADVANCED QUANTUM MULTI-HEAD RAG CIRCUIT
# =============================================================================
@qml.qnode(dev, diff_method="adjoint")
def advanced_rag_attention(params):

    head_size = 6      # 4 heads × 6 qubits = 24
    depth = 4
    idx = 0

    # -------------------------------------------------------------------------
    # INPUT EMBEDDING (Token encoding)
    # -------------------------------------------------------------------------
    for i in range(24):
        qml.RY(params[idx], wires=i); idx += 1
        qml.RZ(params[idx], wires=i); idx += 1

    # -------------------------------------------------------------------------
    # MULTI-HEAD SELF-ATTENTION
    # -------------------------------------------------------------------------
    for h in range(4):
        start = h * head_size
        end = start + head_size

        # Local mixing
        for i in range(start, end-1):
            qml.CNOT(wires=[i, i+1])

        # Trainable attention weights (phase mesh)
        for i in range(start, end):
            for j in range(i+1, end):
                qml.ControlledPhaseShift(params[idx], wires=[i, j])
                idx += 1

    # -------------------------------------------------------------------------
    # CROSS-HEAD ROUTING (Global attention fusion)
    # -------------------------------------------------------------------------
    for i in range(12):
        qml.CNOT(wires=[i, i+12])

    # -------------------------------------------------------------------------
    # TRANSFORMER DEPTH STACK
    # -------------------------------------------------------------------------
    for d in range(depth):
        for i in range(24):
            qml.RX(params[idx], wires=i); idx += 1

        for i in range(23):
            qml.CNOT(wires=[i, i+1])
        qml.CNOT(wires=[23, 0])

    # -------------------------------------------------------------------------
    # OUTPUT HEAD (Adjoint-compatible)
    # -------------------------------------------------------------------------
    return [qml.expval(qml.PauliZ(i)) for i in range(24)]


# =============================================================================
# PARAMETER COUNT
# =============================================================================
embedding_params = 24 * 2
phase_params = 4 * (6 * 5 // 2)   # full pair mesh per head
transformer_params = 4 * 24
total_params = embedding_params + phase_params + transformer_params

print("\nTotal trainable parameters:", total_params)

# IMPORTANT: trainable parameters
params = qml.numpy.random.randn(total_params, requires_grad=True)

# =============================================================================
# BENCHMARK
# =============================================================================
print("\n=== ADVANCED 24-QUBIT ATTENTION BENCHMARK ===")

# Warm-up
print("Warm-up run...")
_ = advanced_rag_attention(params)

# Forward timing
print("Forward pass...")
start = time.time()
out = advanced_rag_attention(params)
forward_time = time.time() - start

print("Forward time:", round(forward_time, 4), "sec")
print("Output sample:", np.round(out[:6], 4))

# Gradient timing
print("\nComputing gradient...")
start = time.time()
grad = qml.grad(advanced_rag_attention)(params)
grad_time = time.time() - start

print("Gradient time:", round(grad_time, 4), "sec")
print("Gradient sample:", np.round(grad[:6], 4))

# Memory usage
mem = psutil.Process().memory_info().rss / 1024**2
print("\nMemory usage:", round(mem, 1), "MiB")

print("\n=== TEST COMPLETE ===")
print("If gradient is non-empty and times are reasonable → full 24Q adjoint GPU success.")

Installing / upgrading packages...
PennyLane version: 0.44.0
GPU Info:
name, memory.total [MiB], utilization.gpu [%]
Tesla T4, 15360 MiB, 0 %

Using device: <lightning.gpu device (wires=24) at 0x7f86224195b0>

Total trainable parameters: 204

=== ADVANCED 24-QUBIT ATTENTION BENCHMARK ===
Warm-up run...


In [ ]:

# =============================================================================
# 20-QUBIT COLOR MIXING WHEEL (8 UNIQUE CIRCUITS)
# =============================================================================

print("Installing packages...")
!pip install --upgrade -q pip setuptools wheel
!pip install -q pennylane pennylane-lightning[gpu] custatevec-cu12

import pennylane as qml
import numpy as np
import time
import psutil

print("PennyLane version:", qml.__version__)
!nvidia-smi --query-gpu=name,memory.total,utilization.gpu --format=csv

# -----------------------------------------------------------------------------
# DEVICE
# -----------------------------------------------------------------------------
dev = qml.device("lightning.gpu", wires=20, batch_obs=True)
print("\nUsing device:", dev)

# -----------------------------------------------------------------------------
# BASE ENTANGLER
# -----------------------------------------------------------------------------
def ring_entangler():
    for i in range(19):
        qml.CNOT(wires=[i, i+1])
    qml.CNOT(wires=[19, 0])

# -----------------------------------------------------------------------------
# 8 UNIQUE COLOR CIRCUITS
# -----------------------------------------------------------------------------
def color_block(params, mode):

    idx = 0

    for i in range(20):

        if mode == 0:      # RED (RX dominant)
            qml.RX(params[idx], wires=i)

        elif mode == 1:    # GREEN (RY dominant)
            qml.RY(params[idx], wires=i)

        elif mode == 2:    # BLUE (RZ dominant)
            qml.RZ(params[idx], wires=i)

        elif mode == 3:    # CYAN (RX+RY)
            qml.RX(params[idx], wires=i)
            qml.RY(params[idx], wires=i)

        elif mode == 4:    # MAGENTA (RX+RZ)
            qml.RX(params[idx], wires=i)
            qml.RZ(params[idx], wires=i)

        elif mode == 5:    # YELLOW (RY+RZ)
            qml.RY(params[idx], wires=i)
            qml.RZ(params[idx], wires=i)

        elif mode == 6:    # WHITE (balanced XYZ)
            qml.RX(params[idx], wires=i)
            qml.RY(params[idx], wires=i)
            qml.RZ(params[idx], wires=i)

        elif mode == 7:    # BLACK (phase-heavy entanglement)
            qml.RZ(params[idx], wires=i)

        idx += 1

    ring_entangler()

    if mode == 7:
        for i in range(0, 20, 2):
            qml.ControlledPhaseShift(np.pi/4, wires=[i, (i+1)%20])

# -----------------------------------------------------------------------------
# MASTER COLOR WHEEL CIRCUIT
# -----------------------------------------------------------------------------
@qml.qnode(dev, diff_method="adjoint")
def color_wheel_surface(params, mix_weights):

    offset = 0

    # Superpose 8 color circuits
    for mode in range(8):
        block_params = params[offset:offset+20]
        color_block(block_params, mode)
        offset += 20

    # Weighted global mixing layer
    for i in range(20):
        qml.RX(mix_weights[0] * i, wires=i)
        qml.RY(mix_weights[1] * i, wires=i)
        qml.RZ(mix_weights[2] * i, wires=i)

    ring_entangler()

    return [qml.expval(qml.PauliZ(i)) for i in range(20)]

# -----------------------------------------------------------------------------
# PARAMS (TRAINABLE)
# -----------------------------------------------------------------------------
color_params = qml.numpy.random.randn(8*20, requires_grad=True)
mix_weights = qml.numpy.random.randn(3, requires_grad=True)

print("\n=== 20Q COLOR WHEEL SURFACE BENCHMARK ===")

# Warm-up
_ = color_wheel_surface(color_params, mix_weights)

# Forward
start = time.time()
out = color_wheel_surface(color_params, mix_weights)
forward_time = time.time() - start

print("Forward time:", round(forward_time, 4), "sec")
print("Output sample:", np.round(out[:6], 4))

# Gradient
start = time.time()
grad = qml.grad(color_wheel_surface, argnum=[0,1])(color_params, mix_weights)
grad_time = time.time() - start

print("Gradient time:", round(grad_time, 4), "sec")
print("Grad sample (color params):", np.round(grad[0][:6], 4))

mem = psutil.Process().memory_info().rss / 1024**2
print("Memory usage:", round(mem,1), "MiB")

print("\n=== COLOR WHEEL TEST COMPLETE ===")

Installing packages...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
PennyLane version: 0.44.0
name, memory.total [MiB], utilization.gpu [%]
Tesla T4, 15360 MiB, 0 %

Using device: <lightning.gpu device (wires=20) at 0x7d70a5330b90>

=== 20Q COLOR WHEEL SURFACE BENCHMARK ===
Forward time: 2.7117 sec
Output sample: [ 0.0003  0.0027 -0.0027  0.0022 -0.0016  0.0001]


/usr/local/lib/python3.12/dist-packages/pennylane/_grad/grad.py:259: PennyLaneDeprecationWarning: argnum in qml.grad has been renamed to argnums to match jax and catalyst.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/autograd/tracer.py:16: UserWarning: Output seems independent of input.
  warnings.warn("Output seems independent of input.")


TypeError: Grad only applies to real scalar-output functions. Try jacobian, elementwise_grad or holomorphic_grad.

In [ ]:

# =============================================================================
# 20-QUBIT COLOR WHEEL SURFACE (FIXED AUTOGRAD VERSION)
# =============================================================================

print("Installing / upgrading packages...")
!pip install --upgrade -q pip setuptools wheel
!pip install -q pennylane pennylane-lightning[gpu] custatevec-cu12

import pennylane as qml
import numpy as np
import time
import psutil

print("PennyLane version:", qml.__version__)
print("GPU Info:")
!nvidia-smi --query-gpu=name,memory.total,utilization.gpu --format=csv

# =============================================================================
# DEVICE
# =============================================================================
dev = qml.device("lightning.gpu", wires=20, batch_obs=True)
print("\nUsing device:", dev)

# =============================================================================
# SHARED ENTANGLER
# =============================================================================
def ring_entangler():
    for i in range(19):
        qml.CNOT(wires=[i, i+1])
    qml.CNOT(wires=[19, 0])

# =============================================================================
# COLOR BLOCK
# =============================================================================
def color_block(params, mode):

    for i in range(20):

        if mode == 0:
            qml.RX(params[i], wires=i)
        elif mode == 1:
            qml.RY(params[i], wires=i)
        elif mode == 2:
            qml.RZ(params[i], wires=i)
        elif mode == 3:
            qml.RX(params[i], wires=i); qml.RY(params[i], wires=i)
        elif mode == 4:
            qml.RX(params[i], wires=i); qml.RZ(params[i], wires=i)
        elif mode == 5:
            qml.RY(params[i], wires=i); qml.RZ(params[i], wires=i)
        elif mode == 6:
            qml.RX(params[i], wires=i)
            qml.RY(params[i], wires=i)
            qml.RZ(params[i], wires=i)
        elif mode == 7:
            qml.RZ(params[i], wires=i)

    ring_entangler()

    if mode == 7:
        for i in range(0, 20, 2):
            qml.ControlledPhaseShift(np.pi/4, wires=[i, (i+1)%20])

# =============================================================================
# MAIN CIRCUIT
# =============================================================================
@qml.qnode(dev, diff_method="adjoint")
def color_wheel_surface(color_params, mix_weights):

    offset = 0

    for mode in range(8):
        block = color_params[offset:offset+20]
        color_block(block, mode)
        offset += 20

    for i in range(20):
        qml.RX(mix_weights[0]*(i+1), wires=i)
        qml.RY(mix_weights[1]*(i+1), wires=i)
        qml.RZ(mix_weights[2]*(i+1), wires=i)

    ring_entangler()

    # IMPORTANT: return autograd-compatible array
    return qml.numpy.array(
        [qml.expval(qml.PauliZ(i)) for i in range(20)]
    )

# =============================================================================
# TRAINABLE PARAMS
# =============================================================================
color_params = qml.numpy.random.randn(160, requires_grad=True)
mix_weights = qml.numpy.random.randn(3, requires_grad=True)

print("\n=== 20Q COLOR WHEEL SURFACE BENCHMARK ===")

# Warm-up
_ = color_wheel_surface(color_params, mix_weights)

# Forward
start = time.time()
out = color_wheel_surface(color_params, mix_weights)
print("Forward time:", round(time.time()-start,4), "sec")
print("Output sample:", np.round(out[:6], 4))

# Jacobian
print("\nComputing Jacobian...")
start = time.time()
jac = qml.jacobian(color_wheel_surface, argnums=[0,1])(
    color_params, mix_weights
)
print("Jacobian time:", round(time.time()-start,4), "sec")

print("Jacobian shape (color params):", jac[0].shape)
print("Jacobian shape (mix weights):", jac[1].shape)

print("Jacobian sample:", np.round(jac[0][0][:6], 4))

print("\nMemory usage:",
      round(psutil.Process().memory_info().rss/1024**2,1), "MiB")

print("\n=== SUCCESS ===")

Installing / upgrading packages...
PennyLane version: 0.44.0
GPU Info:
name, memory.total [MiB], utilization.gpu [%]
Tesla T4, 15360 MiB, 0 %

Using device: <lightning.gpu device (wires=20) at 0x7d70a442fa70>

=== 20Q COLOR WHEEL SURFACE BENCHMARK ===
Forward time: 2.7519 sec
Output sample: [-0.0007  0.0013 -0.0025  0.0004  0.0034 -0.0009]

Computing Jacobian...
Jacobian time: 5.3147 sec
Jacobian shape (color params): (20, 160)
Jacobian shape (mix weights): (20, 3)
Jacobian sample: [-0.0005  0.0005 -0.0012  0.0005  0.0006  0.    ]

Memory usage: 640.9 MiB

=== SUCCESS ===
